In [ ]:
# ==============================================================================
# Cell 1: Imports, Logging, and Kubernetes Authentication
# ==============================================================================
import os
import sys
import json
import glob
import logging
import pandas as pd
from datasets import load_dataset
from kubernetes import client as k8s

# Configure clean logging for the demo
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logging.getLogger("transformers").setLevel(logging.WARNING)
logging.getLogger("datasets").setLevel(logging.WARNING)
logging.getLogger("torch").setLevel(logging.WARNING)

# 🛑 QUICK FIX: Using explicit token auth to bypass ServiceAccount RBAC limits
api_server = "https://api.cluster-dpg8g.dpg8g.sandbox1860.opentlc.com:6443"
token = "sha-"  # Replace with a token from kubeadmin login

configuration = k8s.Configuration()
configuration.host = api_server
configuration.verify_ssl = False
configuration.api_key = {"authorization": f"Bearer {token}"}
api_client = k8s.ApiClient(configuration)

print("✅ Kubernetes explicit token authentication successful.")

# Utility Function
def find_most_recent_checkpoint(output_dir):
    checkpoint_pattern = os.path.join(output_dir, "hf_format", "samples_*.0")
    checkpoint_dirs = glob.glob(checkpoint_pattern)
    if not checkpoint_dirs:
        raise ValueError(f"No checkpoints found in {os.path.join(output_dir, 'hf_format')}")
    return max(checkpoint_dirs, key=os.path.getctime)

In [ ]:
# ==============================================================================
# Cell 2: Global Demo Configuration
# ==============================================================================

# 1. Storage Configuration
PVC_NAME = "shared-support-data"
PVC_MOUNT_PATH = "/opt/app-root/src/shared" # Local workbench mount path
CLUSTER_MOUNT_PATH = "/mnt/shared"          # Path used by distributed worker pods

SA_NAME = "osft-dist-banking-demo" 

with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace", "r") as f:
    NAMESPACE = f.read().strip()

print(f"✅ Operating in Namespace: {NAMESPACE}")
print(f"✅ Using ServiceAccount: {SA_NAME}")

# 3. Extract RHOAI MLflow Variables dynamically
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI")
MLFLOW_K8S = os.environ.get("MLFLOW_K8S_INTEGRATION", "true")
MLFLOW_AUTH = os.environ.get("MLFLOW_TRACKING_AUTH", "kubernetes-namespaced")

if not MLFLOW_URI:
    print("⚠️ MLflow URI not found. Ensure your workbench has the 'opendatahub.io/mlflow-instance' annotation.")
else:
    print(f"✅ MLflow Tracking URI: {MLFLOW_URI}")

In [ ]:
# ==============================================================================
# Cell 3: Data Preparation (The Business Narrative)
# ==============================================================================
print("1️⃣ Downloading Banking77 dataset...")
dataset = load_dataset("PolyAI/banking77")
df_train = dataset['train'].to_pandas()

# Map integer labels to string intent names
label_names = dataset['train'].features['label'].names
df_train['intent_name'] = df_train['label'].apply(lambda x: label_names[x])

print(f"✅ Loaded {len(df_train)} training examples.")

# Setup target directory using our Global Config
output_dir = f"{PVC_MOUNT_PATH}/support-data/train"
os.makedirs(output_dir, exist_ok=True)
output_file = f"{output_dir}/banking77_osft_train.jsonl"

# The Strict JSON System Prompt
system_prompt = """You are an AI routing agent for a digital bank. 
Analyze the customer's message and output a strict JSON object with two keys: 'intent' and 'confidence_routing'. 
Do not include any other text or markdown."""

print("\n2️⃣ Formatting data for OSFT (Strict JSON structure)...")
with open(output_file, "w") as f:
    for index, row in df_train.iterrows():
        target_json = {
            "intent": row['intent_name'],
            "confidence_routing": "high"
        }
        instruction_set = {
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Customer Message: {row['text']}"},
                {"role": "assistant", "content": json.dumps(target_json)}
            ]
        }
        f.write(json.dumps(instruction_set) + "\n")

print(f"✅ Successfully saved formatted OSFT examples to: {output_file}")

In [ ]:
# ==============================================================================
# Cell 4: OSFT Training Hyperparameters
# ==============================================================================
params = {
    # 🤖 Model + Data Paths (Using paths mounted inside the worker pods)
    "model_path": "Qwen/Qwen2.5-1.5B-Instruct",
    "data_path": f"{CLUSTER_MOUNT_PATH}/support-data/train/banking77_osft_train.jsonl",
    "ckpt_output_dir": f"{CLUSTER_MOUNT_PATH}/support-checkpoints",
    "data_output_path": f"{CLUSTER_MOUNT_PATH}/osft-json/_banking_data",
    
    # 🏋️‍♀️ Training Hyperparameters (Tuned for quick demo execution)
    "unfreeze_rank_ratio": 0.25,
    "effective_batch_size": 128,
    "learning_rate": 5.0e-6,
    "num_epochs": 3,
    "lr_scheduler": "cosine",
    "warmup_steps": 0,
    "seed": 42,
    
    # 🏎️ Performance Hyperparameters
    "use_liger": True,
    "max_tokens_per_gpu": 64000,
    "max_seq_len": 2048, 

    # 📊 MLflow Observability Settings
    "mlflow_tracking_uri": MLFLOW_URI,
    "mlflow_experiment_name": "banking77-routing-osft",
    "mlflow_run_name": "qwen-1.5b-run-01",
    
    # 💾 Checkpointing & Infrastructure Settings
    "save_final_checkpoint": True,
    "checkpoint_at_epoch": False,
    "nproc_per_node": 2,
    "nnodes": 2,
}

print("⚙️ Banking Routing Training Hyperparameters Configured")

In [ ]:
# ==============================================================================
# Cell 5: Initialize Kubeflow Trainer Client
# ==============================================================================
from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import TrainerClient
from kubeflow.trainer.rhai import TrainingHubAlgorithms, TrainingHubTrainer

# Disable SSL verification for demo sandbox environments
api_client.configuration.verify_ssl = False
backend_cfg = KubernetesBackendConfig(client_configuration=api_client.configuration)

print("🔌 Connecting to Kubeflow control plane...")
client = TrainerClient(backend_cfg)

# Find the training-hub runtime quietly
th_runtime = next((rt for rt in client.list_runtimes() if rt.name == "training-hub"), None)

if th_runtime:
    print("✅ Successfully connected to Kubeflow and found 'training-hub' runtime.")
else:
    print("❌ 'training-hub' runtime not found. Ensure the RHOAI Training Operator is configured correctly.")

In [ ]:
# ==============================================================================
# Cell 6: Submit Distributed Training Job
# ==============================================================================
from kubeflow.trainer.options.kubernetes import (
    ContainerOverride,
    PodSpecOverride,
    PodTemplateOverride,
    PodTemplateOverrides,
)

print(f"🚀 Submitting Distributed Training Job...")

job_name = client.train(
    trainer=TrainingHubTrainer(
        algorithm=TrainingHubAlgorithms.OSFT,
        func_args=params,
        env={
            "HF_HOME": f"{CLUSTER_MOUNT_PATH}/huggingface",
            "TRITON_CACHE_DIR": f"{CLUSTER_MOUNT_PATH}/.triton",
            "XDG_CACHE_HOME": "/opt/app-root/src/.cache",
            "NCCL_DEBUG": "INFO",
            "MLFLOW_TRACKING_URI": MLFLOW_URI,
            "MLFLOW_K8S_INTEGRATION": str(MLFLOW_K8S),
            "MLFLOW_TRACKING_AUTH": str(MLFLOW_AUTH),
        },
        resources_per_node={
            "nvidia.com/gpu": 2,
            "memory": "64Gi", 
            "cpu": 8,         
        },
    ),
    options=[
        PodTemplateOverrides(
            PodTemplateOverride(
                target_jobs=["node"],
                spec=PodSpecOverride(
                    service_account_name=SA_NAME, # 🔑 Crucial for MLflow RBAC
                    volumes=[
                        {"name": "work", "persistentVolumeClaim": {"claimName": PVC_NAME}},
                        {"name": "shm", "emptyDir": {"medium": "Memory"}}
                    ],
                    containers=[
                        ContainerOverride(
                            name="node",
                            volume_mounts=[
                                {"name": "work", "mountPath": CLUSTER_MOUNT_PATH, "readOnly": False},
                                {"name": "shm", "mountPath": "/dev/shm"}
                            ],
                        )
                    ],
                ),
            )
        )
    ],
    runtime=th_runtime,
)

print(f"✅ Job submitted! Job Name: {job_name}")
# Use client.get_job_logs(job_name) in the next cell to monitor progress!

In [ ]:
# Stream logs
for logline in client.get_job_logs(job_name, follow=True):
    print(logline, end="")

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# testing the base model vs the fine-tuned model running inference in the notebook

base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
test_message = "I tried to use the ATM on 5th street but it ate my debit card! This was really a pleasant experience I enjoyed watching. How do I get it back or order a new one?"

system_prompt = """You are an AI routing agent for a digital bank. 
Analyze the customer's message and output a strict JSON object with two keys: 'intent' and 'confidence_routing'. 
Do not include any other text or markdown."""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": f"Customer Message: {test_message}"}
]

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# --- 1. RUN BASE MODEL INFERENCE ---
print("🤖 Loading Base Model (Qwen2.5-1.5B-Instruct)...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float32,
    device_map="auto"
)

inputs = tokenizer([text], return_tensors="pt").to(base_model.device)

print("Running Base Model Inference...")
with torch.no_grad():
    base_generated_ids = base_model.generate(
        inputs.input_ids,
        max_new_tokens=100,
        temperature=0.1,
        do_sample=True
    )

base_ids = [out[len(inp):] for inp, out in zip(inputs.input_ids, base_generated_ids)]
base_response = tokenizer.batch_decode(base_ids, skip_special_tokens=True)[0]

# Free RAM
del base_model
gc.collect()

# --- 2. RUN FINE-TUNED OSFT MODEL INFERENCE ---
print("\n🎯 Loading OSFT Fine-Tuned Model...")
ft_model = AutoModelForCausalLM.from_pretrained(
    latest_checkpoint,
    torch_dtype=torch.float32,
    device_map="auto"
)

inputs_ft = tokenizer([text], return_tensors="pt").to(ft_model.device)

print("Running Fine-Tuned Model Inference...")
with torch.no_grad():
    ft_generated_ids = ft_model.generate(
        inputs_ft.input_ids,
        max_new_tokens=50,
        temperature=0.1,
        do_sample=True
    )

ft_ids = [out[len(inp):] for inp, out in zip(inputs_ft.input_ids, ft_generated_ids)]
ft_response = tokenizer.batch_decode(ft_ids, skip_special_tokens=True)[0]

# --- 3. DISPLAY COMPARISON ---
print("\n" + "=" * 50)
print("🔴 BASE MODEL RESPONSE:")
print("=" * 50)
print(base_response)

print("\n" + "=" * 50)
print("🟢 OSFT FINE-TUNED MODEL RESPONSE:")
print("=" * 50)
print(ft_response)
print("=" * 50)

In [ ]:
# ==============================================================================
# Cell 9: Config Cleanup for vLLM Serving
# ==============================================================================

def clean_vllm_config(output_dir):
    try:
        # Find the latest checkpoint using our utility from Cell 1
        latest_ckpt = find_most_recent_checkpoint(output_dir)
        config_path = os.path.join(latest_ckpt, "config.json")
        
        print(f"🔍 Inspecting config at: {config_path}")
        
        with open(config_path, 'r') as f:
            config_data = json.load(f)
            
        # The OSFT process occasionally injects a 'quantization_config' that vLLM rejects
        # We strip it out here to ensure clean serving
        if 'quantization_config' in config_data:
            del config_data['quantization_config']
            print("✅ Removed incompatible 'quantization_config' block.")
            
        with open(config_path, 'w') as f:
            json.dump(config_data, f, indent=2)
            
        print("✅ Config sanitized and ready for vLLM deployment!")
        
    except Exception as e:
        print(f"⚠️ Error cleaning config: {e}")

# Run the cleanup on our checkpoint directory
clean_vllm_config(f"{PVC_MOUNT_PATH}/support-checkpoints")

# 🛑 STOP: Manual Deployment Required!
Before running the final cell, you must deploy the model in the RHOAI Dashboard:
1. Go to your Data Science Project.
2. Under **Models**, click **Deploy model** (Select Single-model serving platform).
3. **Model Name:** `banking-routing-model` *(must match exactly)*
4. **Serving Runtime:** `vLLM ServingRuntime`
5. **Model framework:** `vLLM`
6. **Model location:** Existing data connection. Point it to the `shared-support-checkpoints/hf_format/samples_...` folder on your PVC.
7. Wait for the status to show a green checkmark before proceeding!

In [ ]:
# ==============================================================================
# Cell 11: The Solution - Testing the Fine-Tuned Model locally
# ==============================================================================
# Find the exact path of the successful training run
latest_ckpt = find_most_recent_checkpoint(f"{PVC_MOUNT_PATH}/support-checkpoints")
print(f"🤖 Loading Fine-Tuned Model from: {latest_ckpt}")

ft_pipeline = pipeline(
    "text-generation", 
    model=latest_ckpt, 
    device_map="auto", 
    torch_dtype=torch.float16
)

print(f"\n📩 User Input: '{test_message}'\n")
print("⏳ Generating Fine-Tuned Response...")

ft_response = ft_pipeline(
    messages, 
    max_new_tokens=50, 
    return_full_text=False,
    temperature=0.1
)

print("\n✅ Fine-Tuned Model Output (Strict, clean JSON):")
print("-" * 50)
print(ft_response[0]['generated_text'].strip())
print("-" * 50)

# Free up workbench memory
del ft_pipeline
torch.cuda.empty_cache()